In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras.applications import ResNet152, ResNet50, InceptionV3, Xception, VGG19
from keras.layers import Flatten, GlobalAveragePooling2D
from keras.models import Model
from keras.optimizers import Adam
import keras.backend as K
import gc

os.chdir('/media/my_drives')
ImageFile.LOAD_TRUNCATED_IMAGES = True
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
def load_basemodel(MODEL_NAME):
    if MODEL_NAME == 'VGG19':
        return VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    elif MODEL_NAME == 'Inception':
        return InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
    elif MODEL_NAME == 'Xception':
        return Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
    else:
        raise ValueError("Invalid model name.")

def is_image_openable(image_path): # helper function to drop broken images
    try:
        with Image.open(image_path) as img:
            img.verify()
        return True
    except (IOError, SyntaxError):
        return False

def make_datagenerator(train_df, test_df, BATCH_SIZE, IMG_SHAPE, ALL_CLASSES):
    current_batch_size = BATCH_SIZE
    if BATCH_SIZE > test_df.shape[0]:
        current_batch_size = test_df.shape[0]
    
    train_datagen = ImageDataGenerator(rescale=1./255)
    test_datagen = ImageDataGenerator(rescale=1./255)
    
    # Check that labels in the dataframes match the ALL_CLASSES list
    train_df['label'] = train_df['label'].astype(str)
    test_df['label'] = test_df['label'].astype(str)
    
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='image_path',
        y_col='label',
        target_size=IMG_SHAPE,
        batch_size=current_batch_size,
        class_mode='categorical',
        classes=ALL_CLASSES 
    )
    
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='image_path',
        y_col='label',
        target_size=IMG_SHAPE,
        batch_size=current_batch_size,
        class_mode='categorical',
        classes=ALL_CLASSES 
    )
    return train_generator, test_generator

In [ ]:
CSV_PATH = 'DATA4/max/Image_Benchmark/R1_Analytics/Results.csv'
#df = pd.DataFrame(columns=['Model', 'Dataset', 'N_Training_Samples', 'BATCH_SIZE', 'LEARNING_RATE', 'EPOCHS', 'Time_Seconds', 'Accuracy'])
df = pd.read_csv(CSV_PATH)
df.tail(2)

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

In [ ]:
Models = ['VGG19', 'Inception', 'Xception']
BATCH_SIZES = [16, 32, 64]
LEARNING_RATES = [0.01, 0.001, 1e-5]
N_SAMPLES_SIZES = [1000, 200]
EPOCHS = 32

In [ ]:
for DATASET_NAME in datasets: 
    for MODEL_NAME in Models:             
        for BATCH_SIZE in BATCH_SIZES:
            for LEARNING_RATE in LEARNING_RATES:
                for N_Samples in N_SAMPLES_SIZES:
                    try:
                        K.clear_session()
                        train_generator.reset()
                        test_generator.reset()
                    except:
                        pass
                                                
                    if (DATASET_NAME == 'AIDA') & (N_Samples == 1000):
                        continue
                        
                    if df.loc[((df.Model == MODEL_NAME) & (df.Dataset == DATASET_NAME) & (df.N_Training_Samples == N_Samples) & (df.BATCH_SIZE == BATCH_SIZE) & (df.LEARNING_RATE == LEARNING_RATE))].shape[0] == 0:
                        base_model = load_basemodel(MODEL_NAME)
                        for layer in base_model.layers:
                            layer.trainable = False
                            
                        if MODEL_NAME in ['Inception', 'Xception']:
                            IMG_SIZE = (299, 299)
                        else:
                            IMG_SIZE = (224,224)
                         
                        print(MODEL_NAME, ": ", DATASET_NAME, BATCH_SIZE, LEARNING_RATE)
    
                        DATASET_CSV = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_{N_Samples}/{DATASET_NAME}.csv")
                                
                        DATASET_CSV_HOLDOUT = pd.read_csv(f"DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout/{DATASET_NAME}.csv")
                        train_labels = set(DATASET_CSV['label'])
                        DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT[DATASET_CSV_HOLDOUT['label'].isin(train_labels)]
                        label_to_int = {label: idx for idx, label in enumerate(sorted(train_labels))}
                        DATASET_CSV['label'] = DATASET_CSV['label'].map(label_to_int).astype(str)
                        DATASET_CSV_HOLDOUT['label'] = DATASET_CSV_HOLDOUT['label'].map(label_to_int).astype(str)
    
                        class_names = list(DATASET_CSV['label'].unique())
                        n_classes = len(class_names)
                        train_generator, test_generator = make_datagenerator(DATASET_CSV, DATASET_CSV_HOLDOUT, BATCH_SIZE, IMG_SIZE, class_names)
                        if MODEL_NAME == 'VGG19':
                            x = Flatten()(base_model.output)
                        elif MODEL_NAME in ['Inception', 'Xception']: 
                            x = GlobalAveragePooling2D()(base_model.output) 
                        else:
                           raise ValueError("Invalid model name.")
                        predictions = Dense(n_classes, activation='softmax')(x)
                        model = Model(inputs=base_model.input, outputs=predictions)
                    
                        PATIENCE=3
                    
                        if n_classes == 2:
                            LOSS = 'binary_crossentropy'
                        else:
                            LOSS = 'categorical_crossentropy'
                        
                        optimizer = Adam(learning_rate=LEARNING_RATE)
                        early_stopping = EarlyStopping(monitor='val_loss', patience=PATIENCE)
                        model.compile(optimizer=optimizer, loss=LOSS, metrics=['accuracy'])
                        start_time = time.time()
                        metrics = model.fit(train_generator, epochs=EPOCHS, validation_data=test_generator, callbacks=[early_stopping])
                        total_time = np.round(time.time() - start_time, 2)
                        
                        if early_stopping.stopped_epoch != 0 and early_stopping.stopped_epoch != (metrics.epoch[-1]):
                            best_epoch_index = early_stopping.stopped_epoch - PATIENCE
                        else:
                            best_epoch_index = metrics.epoch[-1]
    
                        # fill dataset
                        i = df.shape[0]
                        df.at[i, 'Model'] = MODEL_NAME
                        df.at[i, 'Dataset'] = DATASET_NAME
                        df.at[i, 'N_Training_Samples'] = N_Samples
                        df.at[i, 'BATCH_SIZE'] = BATCH_SIZE
                        df.at[i, 'LEARNING_RATE'] = LEARNING_RATE
                        df.at[i, 'EPOCHS'] = best_epoch_index+1 # index starts by 0, but is 1 epoch
                        df.at[i, 'Time_Seconds'] = total_time
                        df.at[i, 'Accuracy'] = metrics.history['val_accuracy'][best_epoch_index]
                        df.to_csv(CSV_PATH, index=False)

                        K.clear_session()
                        del DATASET_CSV, DATASET_CSV_HOLDOUT
                        del model
                        gc.collect()